## Simulating observations with MUSTANG-2

MUSTANG-2 is a bolometric array on the [Green Bank Telescope](https://en.wikipedia.org/wiki/Green_Bank_Telescope). In this notebook we simulate an observation of the Crab Nebula (M1).

In [ ]:
import maria

input_map = maria.map.get("maps/30dor.fits", nu=150e9).to("uK_RJ")

input_map.plot()
print(input_map)

In [ ]:
from maria import Planner

planner = Planner(target=input_map, site="cerro_chajnantor", constraints={"el": (40, 90)})
plans = planner.generate_plans(total_duration=300, sample_rate=50)

plans[0].plot()
print(plans)

In [ ]:
instrument = maria.get_instrument("apex/artemis")

print(instrument)
instrument.plot()

In [ ]:
sim = maria.Simulation(
    instrument,
    plans=plans,
    site="cerro_chajnantor",
    map=input_map,
    atmosphere="2d",
    atmosphere_kwargs={"weather": {"pwv": 0.25}},
)

print(sim)

In [ ]:
tods = sim.run()
tods[0].plot()

In [ ]:
from maria.mappers import MaximumLikelihoodMapper

mapper = MaximumLikelihoodMapper(
    stokes="I",
    tod_preprocessing={
        "remove_spline": {"knot_spacing": 60, "remove_el_gradient_order": 1},
    },
    tods=tods,
)

mapper.map.plot()

In [ ]:
mapper.fit(epochs=2, steps_per_epoch=25, plot=True)